# Clipping Videos with Transcript Timestamps using Parakeet.

One of the most common usecases for video ai is converting long form content into clips. Lets say you regularly host a 2 hour podcast and want to promote your show using short-form content, highlighting the most exciting or interesting moments of the episode. 



In [ ]:
!pip install "daft[huggingface]" "nemo_toolkit['asr']" av numpy

In [ ]:
uri = "../videoprism/videoprism/assets/*.mp4"
B, T, H, W, C = 2, 16, 288, 288, 3 # Batch Size, Clip Size (# frames), Height, Width, RGB
ROW_LIMIT = 500

In [ ]:
import daft
import numpy as np
from daft import col, DataType as dt
from daft.functions import file, llm_generate
import av
from av.audio.resampler import AudioResampler

In [ ]:
# Files → Metadata
df = (
    daft.from_glob_path(uri)
    .with_column("file", file(col("path")))
)

In [ ]:

@daft.func(return_dtype=dt.struct({
    "index": dt.uint64(),
    "pts": dt.float64(),
}))
def key_frames(
    file: daft.File,
    *,
    probesize: str = "64k",
    analyzeduration_us: int = 200_000,
) -> list[float]:

    options = {
            "probesize": str(probesize),
            "analyzeduration": str(analyzeduration_us),
        }

    with av.open(file,mode="r", options=options, metadata_encoding="utf-8") as container:
        video = next(
            (stream for stream in container.streams if stream.type == "video"),
            None,
        )
        if video is None:
            return {
                "index": [],
                "pts": [],
            }

        fps = None
        if video.average_rate:
            fps = float(video.average_rate)
        elif video.guessed_rate:
            fps = float(video.guessed_rate)

        keyframe_pts = []
        try:
            for packet in container.demux(video):
                if packet.is_keyframe and packet.pts is not None:
                    pts_seconds = float(packet.pts * float(video.time_base))
                    keyframe_pts.append(pts_seconds)
        except Exception:
            keyframe_pts = []

        keyframe_indices = (
            [int(round(t * fps)) for t in keyframe_pts] if fps else []
        )

        return {
            "index": keyframe_indices,
            "pts": keyframe_pts,
        }


In [ ]:
df = df.with_column("key_frames", key_frames(col("file")))

In [ ]:
@daft.func()
def audio(file: daft.File, start_sec: float, end_sec: float, num_frames: int = 16, ) -> np.ndarray:

    container = av.open(file)
    resampler = AudioResampler(format='s16', layout='mono', rate=16000)

    chunks = []
    try:
        for frame in container.decode(audio=0):
            # Resample to desired SR/mono/PCM16; result can be a frame or list of frames
            res = resampler.resample(frame)
            frames = res if isinstance(res, (list, tuple)) else [res]

            for f in frames:
                arr = f.to_ndarray()  # typically (channels, samples) or (samples,)

                # Flatten to 1-D mono
                if arr.ndim == 2:
                    # (1, N) or (N, 1) → (N,)
                    if arr.shape[0] == 1:
                        arr = arr[0]
                    elif arr.shape[1] == 1:
                        arr = arr[:, 0]
                    else:
                        # Unexpected multi-channel after mono resample: average as fallback
                        arr = arr.mean(axis=0)
                elif arr.ndim > 2:
                    arr = arr.reshape(-1)

                # Convert PCM16 → float32 in [-1, 1]
                if arr.dtype != np.float32:
                    arr = (arr.astype(np.float32) / 32768.0).clip(-1.0, 1.0)

                chunks.append(arr)
    finally:
        container.close()

    if not chunks:
        return np.zeros((0,), dtype=np.float32)

    audio = np.concatenate(chunks, axis=0).astype(np.float32, copy=False)
    return audio



In [ ]:
df = df.with_column("audio", audio(col("file")))

In [ ]:
# Parakeet Transcribe with Timestamps
@daft.udf(return_dtype = dt.struct({
    "segment": dt.list(dt.struct({
        "start_offset": dt.int32(),
        "end_offset": dt.int32(),
        "start": dt.float32(),
        "end": dt.float32()
    })),
}))
class ParakeetTranscribeTimestampsUDF:
    def __init__(self, context_size: int = 256):
        import nemo.collections.asr as nemo_asr
        self.asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")
        self.asr_model.change_attention_model(
            self_attention_model="rel_pos_local_attn",
            att_context_size=[context_size, context_size]
        )

    def __call__(self, audio: list[np.ndarray]):
        outputs = self.asr_model.transcribe(audio, timestamps=True)   # No public flag to emit only segments
        return [o.timestamp["segment"] for o in outputs]

In [ ]:
df = df.with_column("transcripts_w_ts", ParakeetTranscribeTimestampsUDF(col("audio")))

In [ ]:
from pydantic import BaseModel, Field

class VideoClip(BaseModel):
    start: float
    end: float
    text: str
    language: str
    audio: np.ndarray
    video: np.ndarray


# Given timestamps and a transcript, generate a list of video clips
@daft.udf(return_dtype=dt.list(dt.struct({
    "start": dt.float32(),
    "end": dt.float32(),
    "text": dt.string()
})))
def generate_video_clips(timestamps: list[dict], transcript: str) -> list[dict]:
    import json
    import re

    # Parse timestamps into a list of dicts
    timestamps_list = [
        {
            "start": float(start),
            "end": float(end),
            "text": text,
            "audio": audio,
            "video": video

        }
        for start, end, text in timestamps
    ]


NameError: name 'daft' is not defined